# Rebuild the CRSP Treasury zero-coupon panel from 1972 to present

This notebook reconstructs a fixed-maturity zero-coupon Treasury yield panel from the **CRSP daily Treasury file** and the **CRSP monthly Treasury file**. It keeps the 17 Diebold-Li maturities used in the classic paper, but extends the rebuilt panel from **1972 through the present day**.

It is set up to:

1. Read the **daily CRSP Treasury CSV in chunks** and keep the **first trading day of each month** for each security.
2. Fall back to the CRSP **monthly** file if you want a faster but less faithful run.
3. Apply CRSP / Fama-Bliss style eligibility and screening logic.
4. Bootstrap an unsmoothed zero curve and interpolate it to the 17 fixed maturities.
5. Export a **1972-present** model-ready panel you can use for DNS, Macro-DNS, and GP work.

This notebook intentionally does **not** include benchmark diagnostics or paper-era comparison code. You asked for the full panel, not another shrine to replication purity.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------
# User paths
# -----------------------------
DAILY_CSV = Path("crsp_treasury_daily_full.csv")   # replace with your full daily file path
MONTHLY_CSV = Path("h3ktxzxmldfnwnbs.csv")        # optional fallback WRDS monthly file
OUTPUT_DIR = Path("./dns_rebuild_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# -----------------------------
# Sample settings
# -----------------------------
SAMPLE_START = pd.Timestamp("1972-01-01")
SAMPLE_END = pd.Timestamp.today().normalize()

# =========================
# Global date settings
# =========================
START_DATE = pd.Timestamp("1972-01-01")
END_DATE = pd.Timestamp.today().normalize()

DNS_MONTHS = np.array([3, 6, 9, 12, 15, 18, 21, 24, 30, 36, 48, 60, 72, 84, 96, 108, 120], dtype=float)
DNS_DAYS = DNS_MONTHS * 30.4375
DNS_COLS = [f"y{int(m)}" for m in DNS_MONTHS]
ABS_YIELD_TOL = 0.002

In [2]:

# ------------------------------------------------------------
# Daily file: keep only the first trading day in each month
# ------------------------------------------------------------

DAILY_USECOLS = [
    "KYTREASNO","KYCRSPID","CRSPID","TCUSIP","TDATDT","TMATDT","IWHY","TCOUPRT","TNIPPY",
    "TVALFC","TFCPDT","IFCPDTF","TFCALDT","TNOTICE","IYMCN","ITYPE","IUNIQ","ITAX","IFLWR",
    "TBANKDT","TSTRIPELIG","TFRGNTGT","CALDT","TDBID","TDASK","TDNOMPRC","TDNOMPRC_FLG",
    "TDSOURCR","TDACCINT","TDRETNUA","TDYLD","TDDURATN","TDPUBOUT","TDTOTOUT","TDPDINT"
]

def first_trading_day_snapshot_from_daily(
    daily_csv: Path,
    start_date: pd.Timestamp = START_DATE,
    end_date: pd.Timestamp = END_DATE,
    chunksize: int = 1_000_000
) -> pd.DataFrame:
    """
    Reads the full daily CRSP Treasury file in chunks and keeps only rows on the
    first trading date of each month across the market. Returns a monthly-style
    snapshot with one row per security per first-trading-day quote.
    """
    first_days = {}
    kept_chunks = []

    # Pass 1: identify first trading day of each month
    for chunk in tqdm(pd.read_csv(daily_csv, usecols=["CALDT"], parse_dates=["CALDT"], chunksize=chunksize), desc="Pass 1: first trading days"):
        chunk = chunk[(chunk["CALDT"] >= start_date) & (chunk["CALDT"] <= end_date)]
        if chunk.empty:
            continue
        chunk["month"] = chunk["CALDT"].dt.to_period("M")
        mins = chunk.groupby("month")["CALDT"].min()
        for month, d in mins.items():
            first_days[month] = min(first_days.get(month, d), d) if month in first_days else d

    first_day_df = pd.Series(first_days, name="first_day").rename_axis("month").reset_index()
    first_day_map = dict(zip(first_day_df["month"], first_day_df["first_day"]))

    # Pass 2: keep only rows from each month's first trading day
    for chunk in tqdm(pd.read_csv(daily_csv, usecols=DAILY_USECOLS, parse_dates=["CALDT","TDATDT","TMATDT","TFCPDT","TFCALDT","TBANKDT"], chunksize=chunksize), desc="Pass 2: keep first-day rows"):
        chunk = chunk[(chunk["CALDT"] >= start_date) & (chunk["CALDT"] <= end_date)]
        if chunk.empty:
            continue
        chunk["month"] = chunk["CALDT"].dt.to_period("M")
        chunk = chunk[chunk["CALDT"] == chunk["month"].map(first_day_map)]
        if chunk.empty:
            continue
        kept_chunks.append(chunk)

    out = pd.concat(kept_chunks, ignore_index=True) if kept_chunks else pd.DataFrame(columns=DAILY_USECOLS)
    out["MCALDT"] = out["CALDT"]
    out = out.drop(columns=["month"], errors="ignore")
    return out

# Uncomment after you download the full daily file:
# daily_firstday = first_trading_day_snapshot_from_daily(DAILY_CSV)
# daily_firstday.to_parquet(OUTPUT_DIR / "daily_firstday_snapshot_1972_2000.parquet", index=False)


In [3]:

# ------------------------------------------------------------
# Optional fallback: monthly file loader
# ------------------------------------------------------------

MONTHLY_USECOLS = [
    "KYTREASNO","KYCRSPID","CRSPID","TCUSIP","TDATDT","TMATDT","IWHY","TCOUPRT","TNIPPY",
    "TVALFC","TFCPDT","IFCPDTF","TFCALDT","TNOTICE","IYMCN","ITYPE","IUNIQ","ITAX","IFLWR",
    "TBANKDT","TSTRIPELIG","TFRGNTGT","MCALDT","TMBID","TMASK","TMNOMPRC","TMNOMPRC_FLG",
    "TMSOURCR","TMACCINT","TMRETNUA","TMYLD","TMDURATN","TMTOTOUT","TMPUBOUT","TMPCYLD",
    "TMRETNXS","TMPDINT"
]

def load_monthly_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=MONTHLY_USECOLS, parse_dates=["MCALDT","TDATDT","TMATDT","TFCPDT","TFCALDT","TBANKDT"])
    df = df[(df["MCALDT"] >= START_DATE) & (df["MCALDT"] <= END_DATE)].copy()
    return df

monthly_raw = load_monthly_file(MONTHLY_CSV)
monthly_raw.head(2)


/tmp/ipykernel_15486/1441425087.py:14: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=MONTHLY_USECOLS, parse_dates=["MCALDT","TDATDT","TMATDT","TFCPDT","TFCALDT","TBANKDT"])


,KYTREASNO,KYCRSPID,CRSPID,TCUSIP,TDATDT,TMATDT,IWHY,TCOUPRT,TNIPPY,TVALFC,TFCPDT,IFCPDTF,TFCALDT,TNOTICE,IYMCN,ITYPE,IUNIQ,ITAX,IFLWR,TBANKDT,TSTRIPELIG,TFRGNTGT,MCALDT,TMBID,TMASK,TMNOMPRC,TMNOMPRC_FLG,TMSOURCR,TMACCINT,TMRETNUA,TMYLD,TMDURATN,TMTOTOUT,TMPUBOUT,TMPCYLD,TMRETNXS,TMPDINT
1017,200788,19720203.4,19720203.4,912793ML,1971-08-05,1972-02-03,1,0.0,0,0.0,NaT,0,NaT,0,NaN,4,0,1,1,NaT,NaN,NaN,1972-01-31,99.97300,99.977165,99.975082,M,R,0.0,0.002895,0.000083,3.0,3903.0,NaN,0.030551,0.000028,0.0
1023,200789,19720210.4,19720210.4,912793MM,1971-08-12,1972-02-10,1,0.0,0,0.0,NaT,0,NaT,0,NaN,4,0,1,1,NaT,NaN,NaN,1972-01-31,99.91111,99.924446,99.917778,M,R,0.0,0.002982,0.000082,10.0,3900.0,NaN,0.030250,0.000105,0.0


In [4]:

# ------------------------------------------------------------
# Harmonize daily snapshot into monthly-style columns
# ------------------------------------------------------------

def daily_to_monthly_style(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename = {
        "CALDT": "MCALDT",
        "TDBID": "TMBID",
        "TDASK": "TMASK",
        "TDNOMPRC": "TMNOMPRC",
        "TDSOURCR": "TMSOURCR",
        "TDACCINT": "TMACCINT",
        "TDRETNUA": "TMRETNUA",
        "TDYLD": "TMYLD",
        "TDDURATN": "TMDURATN",
        "TDPUBOUT": "TMPUBOUT",
        "TDTOTOUT": "TMTOTOUT",
        "TDPDINT": "TMPDINT",
    }
    out = out.rename(columns=rename)
    return out

# Example usage after building the daily snapshot:
# daily_firstday = pd.read_parquet(OUTPUT_DIR / "daily_firstday_snapshot_1972_2000.parquet")
# crsp_input = daily_to_monthly_style(daily_firstday)


In [6]:

# ------------------------------------------------------------
# Core preprocessing
# ------------------------------------------------------------

def make_mid_price(df: pd.DataFrame) -> np.ndarray:
    return np.where(
        (df["TMBID"].fillna(-1) > 0) & (df["TMASK"].fillna(-1) > 0),
        (df["TMBID"] + df["TMASK"]) / 2.0,
        np.where(
            df["TMNOMPRC"].fillna(-1) > 0,
            df["TMNOMPRC"],
            np.where(df["TMBID"].fillna(-1) > 0, df["TMBID"], df["TMASK"])
        )
    )

def prepare_crsp_input(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()

    x["MCALDT"] = pd.to_datetime(x["MCALDT"])
    x["TDATDT"] = pd.to_datetime(x["TDATDT"])
    x["TMATDT"] = pd.to_datetime(x["TMATDT"])
    for c in ["TFCPDT","TFCALDT","TBANKDT"]:
        if c in x.columns:
            x[c] = pd.to_datetime(x[c], errors="coerce")

    x = x[(x["MCALDT"] >= START_DATE) & (x["MCALDT"] <= END_DATE)].copy()

    # CRSP / Fama-Bliss eligible universe
    x = x[
        x["ITYPE"].isin([1, 2, 3, 4]) &
        (x["ITAX"] == 1) &
        (x["IFLWR"] == 1) &
        (x["TNOTICE"] == 0)
    ].copy()

    # Midpoint clean and dirty price
    x["mid_price"] = make_mid_price(x)
    x["dirty_price"] = x["mid_price"] + x["TMACCINT"].fillna(0.0)

    x["days_to_maturity"] = (x["TMATDT"] - x["MCALDT"]).dt.days.astype(float)
    x["years_to_maturity"] = x["days_to_maturity"] / 365.25

    # Diebold-Li extra liquidity screen
    keep = (
        ((x["ITYPE"] == 4) & (x["days_to_maturity"] >= 30)) |
        ((x["ITYPE"] != 4) & (x["days_to_maturity"] >= 365))
    )
    x = x[keep].copy()

    x = x[
        x["mid_price"].notna() &
        (x["mid_price"] > 0) &
        x["dirty_price"].notna() &
        (x["dirty_price"] > 0)
    ].copy()

    # Annualized decimal yield proxy from the CRSP daily/monthly yield field
    x["ytm_ann_dec"] = x["TMYLD"] * 365.0
    x["spread"] = (x["TMASK"] - x["TMBID"]).abs().fillna(np.inf)

    x = x.sort_values(["MCALDT", "days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, True, False, True]).reset_index(drop=True)
    return x


In [7]:

# ------------------------------------------------------------
# Cash-flow machinery
# ------------------------------------------------------------

def remaining_cashflows(row: pd.Series) -> pd.DataFrame:
    quote_date = row["MCALDT"]
    maturity_date = row["TMATDT"]
    coupon_rate = float(row["TCOUPRT"]) / 100.0 if pd.notna(row["TCOUPRT"]) else 0.0
    freq = int(row["TNIPPY"]) if pd.notna(row["TNIPPY"]) else 0

    if (row["ITYPE"] == 4) or (freq == 0) or (coupon_rate == 0.0):
        return pd.DataFrame({
            "date": [maturity_date],
            "t": [(maturity_date - quote_date).days / 365.25],
            "cf": [100.0]
        })

    step_months = int(round(12 / freq))
    dates = [maturity_date]
    d = maturity_date

    while True:
        d = d - pd.DateOffset(months=step_months)
        if d <= quote_date:
            break
        dates.append(d)

    dates = sorted(dates)
    coupon = 100.0 * coupon_rate / freq
    cashflows = [coupon] * len(dates)
    cashflows[-1] += 100.0
    times = [(dt - quote_date).days / 365.25 for dt in dates]

    return pd.DataFrame({"date": dates, "t": times, "cf": cashflows})

def df_from_segments(t, breaks, fwds):
    t = np.asarray(t, dtype=float)
    if len(breaks) == 0:
        return np.ones_like(t, dtype=float)

    cum = np.zeros_like(t)
    prev = 0.0
    for b, f in zip(breaks, fwds):
        overlap = np.clip(np.minimum(t, b) - prev, 0.0, None)
        cum += f * overlap
        prev = b
    return np.exp(-cum)

def solve_incremental_forward(row: pd.Series, prev_T: float, prev_breaks, prev_fwds):
    cf = remaining_cashflows(row)

    known = cf[cf["t"] <= prev_T + 1e-12]
    tail = cf[cf["t"] > prev_T + 1e-12]

    D_prev = float(df_from_segments(np.array([prev_T]), prev_breaks, prev_fwds)[0]) if prev_T > 0 else 1.0
    pv_known = float(np.sum(
        known["cf"].to_numpy() * df_from_segments(known["t"].to_numpy(), prev_breaks, prev_fwds)
    ))

    target = float(row["dirty_price"]) - pv_known
    if len(tail) == 0:
        return np.nan

    delta = tail["t"].to_numpy() - prev_T
    cfs = tail["cf"].to_numpy()

    def objective(fwd):
        return D_prev * np.sum(cfs * np.exp(-fwd * delta)) - target

    lo, hi = -0.10, 0.30
    flo, fhi = objective(lo), objective(hi)
    tries = 0
    while flo * fhi > 0 and tries < 20:
        lo -= 0.05
        hi += 0.05
        flo, fhi = objective(lo), objective(hi)
        tries += 1
    if flo * fhi > 0:
        return np.nan

    return brentq(objective, lo, hi, maxiter=500)


In [8]:

# ------------------------------------------------------------
# Pass 1 / 2 / 3 / 4 logic
# ------------------------------------------------------------

def _window_mean(vals):
    return np.nan if len(vals) == 0 else max(0.0, float(np.mean(vals)))

def first_pass(month_df: pd.DataFrame):
    g = month_df.sort_values(["days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, False, True]).reset_index(drop=True).copy()
    if len(g) <= 6:
        return g.copy(), g.iloc[0:0].copy()

    include = np.zeros(len(g), dtype=bool)

    for i in range(len(g)):
        row = g.iloc[i]
        y = row["ytm_ann_dec"]

        shorter = g.iloc[:i].copy()
        longer = g.iloc[i + 1:].copy()

        # Bills-only windows as long as bills exist
        if len(shorter[shorter["ITYPE"] == 4]) >= 3:
            shorter = shorter[shorter["ITYPE"] == 4]
        if len(longer[longer["ITYPE"] == 4]) >= 3:
            longer = longer[longer["ITYPE"] == 4]

        # Exclude 1.5% notes in pass 1 windows
        shorter = shorter[shorter["TCOUPRT"] != 1.5].tail(3)
        longer = longer[longer["TCOUPRT"] != 1.5].head(3)

        if len(shorter) == 0 or len(longer) == 0:
            continue

        sbar = _window_mean(shorter["ytm_ann_dec"].to_numpy())
        lbar = _window_mean(longer["ytm_ann_dec"].to_numpy())
        lo, hi = sorted([sbar, lbar])

        if (abs(y - sbar) <= ABS_YIELD_TOL) or (abs(y - lbar) <= ABS_YIELD_TOL) or (lo <= y <= hi):
            include[i] = True

    include[-1] = True  # longest maturity always included
    kept = g.loc[include].copy()

    # Resolve maturity clustering under 7-day spacing
    resolved = []
    def pref_tuple(r):
        return (0 if r["ITYPE"] == 4 else 1, r["spread"], abs(r["mid_price"] - 100.0))

    for _, r in kept.iterrows():
        if not resolved:
            resolved.append(r)
            continue
        prev = resolved[-1]
        same_maturity = int(r["days_to_maturity"]) == int(prev["days_to_maturity"])
        too_close = abs(r["days_to_maturity"] - prev["days_to_maturity"]) < 7
        if same_maturity:
            resolved.append(r)
        elif too_close:
            if pref_tuple(r) < pref_tuple(prev):
                resolved[-1] = r
        else:
            resolved.append(r)

    kept = pd.DataFrame(resolved).reset_index(drop=True)
    excluded = g.loc[~g["TCUSIP"].isin(kept["TCUSIP"])].copy()
    return kept, excluded

def bootstrap_term_structure(accepted_df: pd.DataFrame):
    if accepted_df.empty:
        return pd.DataFrame(), accepted_df.copy()

    g = accepted_df.sort_values(["days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, False, True]).copy()
    g["TMATDT"] = pd.to_datetime(g["TMATDT"])
    g["maturity_key"] = g["TMATDT"]

    breaks, fwds = [], []
    prev_T = 0.0
    curve_rows, bond_rows = [], []

    for mat_date, group in g.groupby("maturity_key", sort=True):
        T_i = float(group["days_to_maturity"].iloc[0] / 365.25)
        f_candidates, z_candidates = [], []

        for _, row in group.iterrows():
            fi = solve_incremental_forward(row, prev_T, breaks, fwds)
            if np.isfinite(fi):
                f_candidates.append(fi)
                D_prev = float(df_from_segments(np.array([prev_T]), breaks, fwds)[0]) if prev_T > 0 else 1.0
                D_i = D_prev * np.exp(-fi * (T_i - prev_T))
                zi = -np.log(D_i) / T_i
                z_candidates.append(zi)

                bond_rows.append({
                    "TCUSIP": row["TCUSIP"],
                    "TMATDT": row["TMATDT"],
                    "days_to_maturity": row["days_to_maturity"],
                    "pass_forward": fi,
                    "pass_zero_yield": zi
                })

        if len(f_candidates) == 0:
            continue

        f_bar = float(np.mean(f_candidates))
        breaks.append(T_i)
        fwds.append(f_bar)

        D_bar = float(df_from_segments(np.array([T_i]), breaks, fwds)[0])
        z_bar = -np.log(D_bar) / T_i

        curve_rows.append({
            "TMATDT": mat_date,
            "days_to_maturity": group["days_to_maturity"].iloc[0],
            "n_issues": len(group),
            "forward_cc": f_bar,
            "zero_yield_cc": z_bar
        })

        prev_T = T_i

    curve_df = pd.DataFrame(curve_rows)
    bond_curve_df = pd.DataFrame(bond_rows)

    if bond_curve_df.empty:
        out = g.copy()
        out["pass_forward"] = np.nan
        out["pass_zero_yield"] = np.nan
        return curve_df, out

    out = g.merge(bond_curve_df, on=["TCUSIP", "TMATDT", "days_to_maturity"], how="left")
    return curve_df, out

def _find_reversal_sequences(y):
    y = np.asarray(y, dtype=float)
    dy = np.diff(y)
    seqs = []
    i = 0
    while i < len(dy) - 1:
        if abs(dy[i]) <= ABS_YIELD_TOL:
            i += 1
            continue
        sign = np.sign(dy[i])
        j = i + 1
        if j >= len(dy):
            break
        if (abs(dy[j]) > ABS_YIELD_TOL) and (np.sign(dy[j]) == -sign):
            k = j
            while k + 1 < len(dy) and abs(dy[k + 1]) > ABS_YIELD_TOL and np.sign(dy[k + 1]) == sign:
                k += 1
            seqs.append((i, k + 1))
            i = k + 1
        else:
            i += 1
    return seqs

def second_or_fourth_pass(accepted_df: pd.DataFrame):
    if accepted_df.empty:
        return accepted_df.copy(), accepted_df.iloc[0:0].copy(), pd.DataFrame()

    curve_df, bond_curve = bootstrap_term_structure(accepted_df)
    if curve_df.empty:
        return accepted_df.copy(), accepted_df.iloc[0:0].copy(), curve_df

    rep = (
        bond_curve.groupby("TMATDT", as_index=False)
        .agg(days_to_maturity=("days_to_maturity", "first"), zero_yield_cc=("pass_zero_yield", "mean"))
        .sort_values("days_to_maturity")
        .reset_index(drop=True)
    )

    seqs = _find_reversal_sequences(rep["zero_yield_cc"].to_numpy())
    delete_mats = set()

    for start_edge, end_edge in seqs:
        positions = list(range(start_edge + 1, end_edge + 1))
        for idx_from_end in range(1, len(positions), 2):
            delete_mats.add(rep.loc[positions[-(idx_from_end + 1)], "TMATDT"])

    kept = accepted_df[~accepted_df["TMATDT"].isin(delete_mats)].copy()
    dropped = accepted_df[accepted_df["TMATDT"].isin(delete_mats)].copy()
    return kept, dropped, curve_df

def implied_zero_for_candidate(row: pd.Series, accepted_df: pd.DataFrame):
    curve_df, _ = bootstrap_term_structure(accepted_df)
    if curve_df.empty:
        breaks, fwds = [], []
        prev_T = 0.0
    else:
        breaks_all = curve_df["days_to_maturity"].to_numpy() / 365.25
        fwds_all = curve_df["forward_cc"].to_numpy()
        prev_candidates = curve_df[curve_df["TMATDT"] < row["TMATDT"]]
        prev_T = float(prev_candidates["days_to_maturity"].max() / 365.25) if len(prev_candidates) else 0.0
        mask = breaks_all <= prev_T + 1e-12
        breaks = list(breaks_all[mask])
        fwds = list(fwds_all[:np.sum(mask)])

    fi = solve_incremental_forward(row, prev_T, breaks, fwds)
    if not np.isfinite(fi):
        return np.nan

    T = float(row["days_to_maturity"] / 365.25)
    D_prev = float(df_from_segments(np.array([prev_T]), breaks, fwds)[0]) if prev_T > 0 else 1.0
    D = D_prev * np.exp(-fi * (T - prev_T))
    return -np.log(D) / T

def third_pass(pass2_kept: pd.DataFrame, pass12_excluded: pd.DataFrame):
    accepted = pass2_kept.sort_values(["days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, False, True]).copy()
    excluded = pass12_excluded.sort_values(["days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, False, True]).copy()

    put_back = []
    for _, row in excluded.iterrows():
        current_curve, current_bond_curve = bootstrap_term_structure(accepted)
        cand_T = row["days_to_maturity"]

        if current_curve.empty:
            shorter = pd.DataFrame()
            longer = pd.DataFrame()
        else:
            current_bond_curve = current_bond_curve.sort_values(["days_to_maturity", "TCOUPRT", "TCUSIP"], ascending=[True, False, True])
            shorter = current_bond_curve[current_bond_curve["days_to_maturity"] < cand_T].tail(3)
            longer = current_bond_curve[current_bond_curve["days_to_maturity"] > cand_T].head(3)

        if len(shorter) < 3 or len(longer) < 3:
            continue

        z_cand = implied_zero_for_candidate(row, accepted)
        if not np.isfinite(z_cand):
            continue

        sbar = _window_mean(shorter["pass_zero_yield"].to_numpy())
        lbar = _window_mean(longer["pass_zero_yield"].to_numpy())
        lo, hi = sorted([sbar, lbar])

        if (abs(z_cand - sbar) <= ABS_YIELD_TOL) or (abs(z_cand - lbar) <= ABS_YIELD_TOL) or (lo <= z_cand <= hi):
            accepted = pd.concat([accepted, row.to_frame().T], ignore_index=True)
            accepted["TMATDT"] = pd.to_datetime(accepted["TMATDT"])
            accepted["MCALDT"] = pd.to_datetime(accepted["MCALDT"])
            accepted["days_to_maturity"] = accepted["days_to_maturity"].astype(float)
            put_back.append(row)

    put_back = pd.DataFrame(put_back) if put_back else excluded.iloc[0:0].copy()
    still_excluded = excluded[~excluded["TCUSIP"].isin(put_back["TCUSIP"])].copy() if len(put_back) else excluded.copy()
    return accepted, still_excluded, put_back


In [9]:

# ------------------------------------------------------------
# Final interpolation and panel build
# ------------------------------------------------------------

def interpolate_dns_from_curve(curve_df: pd.DataFrame):
    curve_df = curve_df.sort_values("days_to_maturity")
    x = curve_df["days_to_maturity"].to_numpy(dtype=float)
    y = curve_df["zero_yield_cc"].to_numpy(dtype=float)

    vals = np.interp(DNS_DAYS, x, y, left=np.nan, right=np.nan)
    if x.min() > DNS_DAYS.min():
        vals[DNS_DAYS < x.min()] = np.nan
    if x.max() < DNS_DAYS.max():
        vals[DNS_DAYS > x.max()] = np.nan

    return dict(zip(DNS_COLS, vals))

def build_dns_panel(crsp_input: pd.DataFrame):
    base = prepare_crsp_input(crsp_input)

    month_outputs = []
    accepted_bonds_all = []
    curve_points_all = []

    for mdate, month_df in tqdm(base.groupby("MCALDT", sort=True), total=base["MCALDT"].nunique(), desc="Build monthly panel"):
        p1_kept, p1_excl = first_pass(month_df)
        p2_kept, p2_drop, curve_p2 = second_or_fourth_pass(p1_kept)
        pass12_excl = pd.concat([p1_excl, p2_drop], ignore_index=True).drop_duplicates(subset=["TCUSIP", "TMATDT"])
        p3_kept, p3_excl, p3_putback = third_pass(p2_kept, pass12_excl)
        p4_kept, p4_drop, curve_p4 = second_or_fourth_pass(p3_kept)
        final_curve, final_bond_curve = bootstrap_term_structure(p4_kept)

        row = {
            "date": mdate,
            "n_raw": len(month_df),
            "n_pass1": len(p1_kept),
            "n_pass2": len(p2_kept),
            "n_pass3": len(p3_kept),
            "n_pass4": len(p4_kept),
        }
        if not final_curve.empty:
            row.update(interpolate_dns_from_curve(final_curve))
        else:
            row.update({c: np.nan for c in DNS_COLS})
        month_outputs.append(row)

        kept = p4_kept.copy()
        kept["date"] = mdate
        accepted_bonds_all.append(kept)

        if not final_curve.empty:
            cp = final_curve.copy()
            cp["date"] = mdate
            curve_points_all.append(cp)

    dns_panel = pd.DataFrame(month_outputs).sort_values("date").reset_index(drop=True)
    accepted_bonds = pd.concat(accepted_bonds_all, ignore_index=True) if accepted_bonds_all else pd.DataFrame()
    curve_points = pd.concat(curve_points_all, ignore_index=True) if curve_points_all else pd.DataFrame()

    return dns_panel, accepted_bonds, curve_points


In [10]:

# ------------------------------------------------------------
# Choose input source
# ------------------------------------------------------------

# OPTION A: preferred
# 1) build daily first-trading-day snapshot from the full daily CSV
# daily_firstday = first_trading_day_snapshot_from_daily(DAILY_CSV)
# crsp_input = daily_to_monthly_style(daily_firstday)

# OPTION B: temporary fallback using WRDS monthly file
crsp_input = monthly_raw.copy()

# To use the preferred daily route, uncomment the two lines below and comment out the fallback:
# daily_firstday = first_trading_day_snapshot_from_daily(DAILY_CSV)
# crsp_input = daily_to_monthly_style(daily_firstday)

print(crsp_input.shape)
crsp_input.head(2)


(161399, 37)


,KYTREASNO,KYCRSPID,CRSPID,TCUSIP,TDATDT,TMATDT,IWHY,TCOUPRT,TNIPPY,TVALFC,TFCPDT,IFCPDTF,TFCALDT,TNOTICE,IYMCN,ITYPE,IUNIQ,ITAX,IFLWR,TBANKDT,TSTRIPELIG,TFRGNTGT,MCALDT,TMBID,TMASK,TMNOMPRC,TMNOMPRC_FLG,TMSOURCR,TMACCINT,TMRETNUA,TMYLD,TMDURATN,TMTOTOUT,TMPUBOUT,TMPCYLD,TMRETNXS,TMPDINT
1017,200788,19720203.4,19720203.4,912793ML,1971-08-05,1972-02-03,1,0.0,0,0.0,NaT,0,NaT,0,NaN,4,0,1,1,NaT,NaN,NaN,1972-01-31,99.97300,99.977165,99.975082,M,R,0.0,0.002895,0.000083,3.0,3903.0,NaN,0.030551,0.000028,0.0
1023,200789,19720210.4,19720210.4,912793MM,1971-08-12,1972-02-10,1,0.0,0,0.0,NaT,0,NaT,0,NaN,4,0,1,1,NaT,NaN,NaN,1972-01-31,99.91111,99.924446,99.917778,M,R,0.0,0.002982,0.000082,10.0,3900.0,NaN,0.030250,0.000105,0.0


In [ ]:

# ------------------------------------------------------------
# Run the reconstruction
# ------------------------------------------------------------

dns_panel, accepted_bonds, curve_points = build_dns_panel(crsp_input)

dns_panel.to_csv(OUTPUT_DIR / "dns_panel_rebuilt_1972_present.csv", index=False)
accepted_bonds.to_csv(OUTPUT_DIR / "accepted_bonds_rebuilt_1972_present.csv", index=False)
curve_points.to_csv(OUTPUT_DIR / "curve_points_rebuilt_1972_present.csv", index=False)

dns_panel.head()


Build monthly panel:   0%|          | 0/648 [00:00<?, ?it/s]

/tmp/ipykernel_15486/3520442707.py:236: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  accepted = pd.concat([accepted, row.to_frame().T], ignore_index=True)
/tmp/ipykernel_15486/3520442707.py:236: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  accepted = pd.concat([accepted, row.to_frame().T], ignore_index=True)
/tmp/ipykernel_15486/3520442707.py:236: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude 

In [ ]:

# ------------------------------------------------------------
# Export a model-ready dataset in percent units
# ------------------------------------------------------------

model_ready = dns_panel.copy()
model_ready["obs"] = model_ready["date"].dt.to_period("M").astype(str).str.replace("-", "M", regex=False)

export = model_ready[["obs", "date"] + DNS_COLS].copy()
for c in DNS_COLS:
    export[c] = 100 * export[c]

rename = {f"y{int(m)}": f"M{int(m)}" for m in DNS_MONTHS}
export = export.rename(columns=rename)

export.to_csv(OUTPUT_DIR / "dns_panel_model_ready_percent_1972_present.csv", index=False)
export.head()


## What to do next

1. Replace `DAILY_CSV` with the path to your **full** daily Treasury CSV.
2. Use the **daily snapshot** route for the main build.
3. If you only need a fast fallback, switch `crsp_input` to `monthly_raw.copy()`.
4. Feed the exported model-ready panel into your DNS / Macro-DNS / GP workflow.
